# NYC Hydrant Coverage Analysis with GeoPandas

This notebook loads the two NYC datasets from the Open Data Program (neighborhoods and fire hydrants),
explores them with GeoPandas, performs spatial analysis, and exports the results.

## 1. Setup and Imports

In [8]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_columns', None)

print(f"GeoPandas version: {gpd.__version__}")

GeoPandas version: 1.1.4


# 2. Load NYC Neighborhoods

In [9]:
neighborhoods = gpd.read_file("./data/raw/nyc_neighborhoods.geojson")

print(f"Shape: {neighborhoods.shape}")
print(f"CRS:   {neighborhoods.crs}")
print(f"Geometry type(s): {neighborhoods.geom_type.unique()}")
neighborhoods.head()

Shape: (262, 12)
CRS:   EPSG:4326
Geometry type(s): <ArrowStringArray>
['MultiPolygon']
Length: 1, dtype: str


,shape_area,ntaname,cdtaname,shape_leng,boroname,ntatype,nta2020,borocode,countyfips,ntaabbrev,cdta2020,geometry
0,35321808.4385,Greenpoint,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),28919.5607283,Brooklyn,0,BK0101,3,047,Grnpt,BK01,"MULTIPOLYGON (((-73.93213 40.72816, -73.93238 ..."
1,28852852.9485,Williamsburg,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),28134.0825628,Brooklyn,0,BK0102,3,047,Wllmsbrg,BK01,"MULTIPOLYGON (((-73.95814 40.7244, -73.95772 4..."
2,15208960.645,South Williamsburg,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),18250.2800908,Brooklyn,0,BK0103,3,047,SWllmsbrg,BK01,"MULTIPOLYGON (((-73.95024 40.70547, -73.94984 ..."
3,52267406.735,East Williamsburg,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),43184.8003755,Brooklyn,0,BK0104,3,047,EWllmsbrg,BK01,"MULTIPOLYGON (((-73.92406 40.71411, -73.92404 ..."
4,9982022.78507,Brooklyn Heights,BK02 Downtown Brooklyn-Fort Greene (CD 2 Appro...,14312.1924247,Brooklyn,0,BK0201,3,047,BkHts,BK02,"MULTIPOLYGON (((-73.99236 40.68969, -73.99436 ..."


# 3. Load NYC Fire Hydrants

In [7]:
hydrants = gpd.read_file("./data/raw/nyc_hydrants.geojson")

print(f"Shape: {hydrants.shape}")
print(f"CRS:   {hydrants.crs}")
print(f"Geometry type(s): {hydrants.geom_type.unique()}")
hydrants.head()

Shape: (109725, 8)
CRS:   EPSG:4326
Geometry type(s): <ArrowStringArray>
['Point']
Length: 1, dtype: str


,cb,latitude,unitid,point_y,longitude,point_x,boro,geometry
0,407,40.7722168,H425919a,220683.273,-73.79457092,1041150.586,4,POINT (-73.79457 40.77222)
1,318,40.64434814,H325449,174041.245,-73.9128952,1008423.396,3,POINT (-73.91289 40.64435)
2,301,40.72505569,H307276,203437.906,-73.95304108,997266.219,3,POINT (-73.95304 40.72506)
3,302,40.6939888,H301843,192115.373,-73.99462891,985738.421,3,POINT (-73.99463 40.69399)
4,402,40.73529053,H439410,207168.6435,-73.93569183,1002071.97,4,POINT (-73.93569 40.73529)


# 4. Filter. Select all neighborhoods in Manhattan

Sanity check the data is loaded 

In [11]:
manhattan_neighborhoods = neighborhoods[neighborhoods['boroname'] == 'Manhattan']

manhattan_neighborhoods.head(5)

,shape_area,ntaname,cdtaname,shape_leng,boroname,ntatype,nta2020,borocode,countyfips,ntaabbrev,cdta2020,geometry
118,19226758.9064,Financial District-Battery Park City,MN01 Financial District-Tribeca (CD 1 Equivalent),44672.4861597,Manhattan,0,MN0101,1,061,FiDi,MN01,"MULTIPOLYGON (((-74.00078 40.69429, -74.00096 ..."
119,13578197.7714,Tribeca-Civic Center,MN01 Financial District-Tribeca (CD 1 Equivalent),21219.504238,Manhattan,0,MN0102,1,061,TriBeCa,MN01,"MULTIPOLYGON (((-73.99931 40.71755, -73.99945 ..."
120,11993281.8636,The Battery-Governors Island-Ellis Island-Libe...,MN01 Financial District-Tribeca (CD 1 Equivalent),49616.7910027,Manhattan,9,MN0191,1,061,Btry_GvIsl,MN01,"MULTIPOLYGON (((-74.01093 40.68449, -74.01193 ..."
121,12916761.5131,SoHo-Little Italy-Hudson Square,MN02 Greenwich Village-SoHo (CD 2 Equivalent),18577.111437,Manhattan,0,MN0201,1,061,SoHo,MN02,"MULTIPOLYGON (((-74.00282 40.72836, -74.00272 ..."
122,10602995.4532,Greenwich Village,MN02 Greenwich Village-SoHo (CD 2 Equivalent),12975.0711639,Manhattan,0,MN0202,1,061,GrnwchVlg,MN02,"MULTIPOLYGON (((-73.9899 40.73443, -73.98987 4..."


#  5. Spatial join. 
Use ST_Contains to match each hydrant to the neighborhood that contains it.

In [13]:
hydrants_with_nta = gpd.sjoin(
    hydrants,
    manhattan_neighborhoods[['ntaname', 'boroname', 'geometry']],
    how='inner',
    predicate='within',
)

print(f"Hydrants matched to a neighborhood: {len(hydrants_with_nta):,}")
print(f"Hydrants NOT matched (outside boundaries): {len(hydrants) - len(hydrants_with_nta):,}")
hydrants_with_nta.head()

Hydrants matched to a neighborhood: 13,305
Hydrants NOT matched (outside boundaries): 96,420


,cb,latitude,unitid,point_y,longitude,point_x,boro,geometry,index_right,ntaname,boroname
688,108,40.76268387,H439894,217146.84,-73.94865417,998473.924,1,POINT (-73.94865 40.76268),139,Upper East Side-Lenox Hill-Roosevelt Island,Manhattan
1376,108,40.76608276,H440032,218386.18325,-73.94472504,999561.82475,1,POINT (-73.94472 40.76608),139,Upper East Side-Lenox Hill-Roosevelt Island,Manhattan
2573,108,40.76764297,H440110,218954.24525,-73.94355011,999886.1915,1,POINT (-73.94355 40.76764),139,Upper East Side-Lenox Hill-Roosevelt Island,Manhattan
3026,108,40.76298904,H439962,217258.213,-73.94898987,998381.3,1,POINT (-73.94899 40.76299),139,Upper East Side-Lenox Hill-Roosevelt Island,Manhattan
4316,108,40.75780487,H440542,215369.02275,-73.95495605,996728.88675,1,POINT (-73.95496 40.75781),139,Upper East Side-Lenox Hill-Roosevelt Island,Manhattan
